# ChuckleNet WavLM + Prosody Extraction - 555 Videos

## Full Utterance-Level Extraction for Scale-Up

This Colab notebook:
1. Mounts Google Drive to access audio files
2. Loads `utterances_clean.jsonl` from GitHub
3. Extracts WavLM (768-dim) + Prosody (21-dim) at **utterance level**
4. Saves combined dataset to Google Drive
5. Trains and validates the model

**Key Fix:** Uses utterance boundaries from Whisper, NOT fixed intervals.
**Expected:** 20%+ positive rate (vs broken 0.7% from fixed intervals)

**Runtime:** ~2-3 hours on T4 GPU

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

## Step 2: Install Packages

In [ ]:
!pip install -q transformers librosa scikit-learn

import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import librosa
import time
from pathlib import Path

SR = 16000
MAX_DURATION = 5.0
BATCH = 16
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Step 3: Load Utterances from GitHub

In [ ]:
!wget -q -O /tmp/utterances_clean.jsonl.gz https://github.com/Das-rebel/chuck-audio-notebooks/raw/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utterances_clean.jsonl.gz

utterances = []
with open('/tmp/utterances_clean.jsonl') as f:
    for line in f:
        utterances.append(json.loads(line))

print(f'Loaded {len(utterances)} utterances')

# Build video -> utterances mapping
vtt_data = {}
for u in utterances:
    vid = u['video_id']
    if vid not in vtt_data:
        vtt_data[vid] = []
    vtt_data[vid].append(u)

print(f'Unique videos: {len(vtt_data)}')

# Label distribution
total_laugh = sum(1 for u in utterances if u['label'] == 1)
print(f'Positive rate: {total_laugh}/{len(utterances)} ({total_laugh/len(utterances)*100:.2f}%)')

## Step 4: Find Audio Files on Google Drive

In [ ]:
# Audio file locations on Google Drive
# Check multiple possible locations
audio_base = Path('/content/gdrive/MyDrive')

# Build audio file map
audio_files = {}
search_folders = [
    audio_base / 'chuckle_audio',
    audio_base / 'chuckle_audio_all' / 'audio',
    audio_base / 'chuckle_audio_all' / 'audio_all',
    audio_base / 'chuckle_audio_all' / 'audio_final',
    audio_base / 'chuckle_audio_all' / 'audio_new',
    audio_base / 'audio',  # Alternative location
    audio_base / 'data' / 'audio',
]

for folder in search_folders:
    if folder.exists():
        print(f'Searching: {folder}')
        for f in folder.glob('*'):
            if f.suffix in ['.m4a', '.mp3', '.wav']:
                vid = f.stem
                audio_files[vid] = str(f)

print(f'\nFound {len(audio_files)} audio files')

# Check coverage
covered = sum(1 for vid in vtt_data if vid in audio_files)
print(f'Utterances with audio: {covered}/{len(vtt_data)} videos')

## Step 5: Load WavLM Model

In [ ]:
print('Loading WavLM model...')
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
model = model.to(device).eval()
print('Model loaded!')

## Step 6: Define Extraction Functions

In [ ]:
def extract_prosody(audio_path, start, end, sr=SR):
    """Extract 21-dim prosody features for an audio segment."""
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
        if len(y) < sr * 0.05:
            return np.zeros(21, dtype=np.float32)
        
        feats = []
        
        # 1. F0/pitch features (5 dims)
        try:
            f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
            f0_values = f0[~np.isnan(f0)]
            if len(f0_values) > 0:
                feats.extend([np.mean(f0_values), np.std(f0_values), np.max(f0_values), np.min(f0_values), np.mean(voiced_flag)])
            else:
                feats.extend([0, 0, 0, 0, 0])
        except:
            feats.extend([0, 0, 0, 0, 0])
        
        # 2. Energy features (5 dims)
        try:
            rms = librosa.feature.rms(y=y)[0]
            feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
        except:
            feats.extend([0, 0, 0, 0, 0])
        
        # 3. Duration features (2 dims)
        duration = len(y) / sr
        speech_rate = 1 / duration if duration > 0 else 0
        feats.extend([duration, speech_rate])
        
        # 4. Spectral features (5 dims)
        try:
            spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
            spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
            spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)[0]
            spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]
            feats.extend([np.mean(spectral_centroid), np.mean(spectral_bandwidth), np.mean(spectral_rolloff), np.mean(spectral_contrast), np.mean(spectral_flatness)])
        except:
            feats.extend([0, 0, 0, 0, 0])
        
        # 5. Voice quality features (4 dims)
        try:
            zcr = librosa.feature.zero_crossing_rate(y)[0]
            jitter = np.mean(np.abs(np.diff(f0_values))) / np.mean(f0_values) if len(f0_values) > 1 else 0
            shimmer = np.mean(np.abs(np.diff(rms))) / np.mean(rms) if len(rms) > 1 else 0
            feats.extend([np.mean(zcr), jitter, shimmer, np.mean(voiced_flag)])
        except:
            feats.extend([0, 0, 0, 0])
        
        return np.array(feats[:21], dtype=np.float32)
    except:
        return np.zeros(21, dtype=np.float32)

def load_audio_segment(video_id, start, end, audio_files, max_duration=MAX_DURATION):
    """Load audio segment for a specific utterance."""
    if video_id not in audio_files:
        return None, None
    
    audio_path = audio_files[video_id]
    try:
        offset = max(0, start - 0.05)
        y, sr = librosa.load(audio_path, sr=SR, mono=True, offset=offset, duration=max_duration)
        target_len = int(MAX_DURATION * SR)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]
        return y, sr
    except:
        return None, None

def extract_wavlm(waveform, model, device):
    """Extract WavLM embedding using attention pooling."""
    if waveform is None or len(waveform) < 160:
        return np.zeros(768, dtype=np.float32)
    
    try:
        inputs = torch.FloatTensor(waveform).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(inputs)
            hidden = outputs.last_hidden_state.squeeze(0).cpu().numpy()
        
        # Attention pooling
        attn_weights = torch.softmax(torch.nn.Linear(768, 1)(torch.tensor(hidden)), dim=0).numpy()
        emb = np.dot(attn_weights.T, hidden).squeeze()
        
        return emb.astype(np.float32)
    except:
        return np.zeros(768, dtype=np.float32)

## Step 7: Extract Features

In [ ]:
print('Starting extraction...')
print(f'Processing {len(utterances)} utterances from {len(vtt_data)} videos')

all_embeddings = []
all_prosody = []
all_labels = []
all_uids = []

start_time = time.time()
errors = 0

for i, u in enumerate(utterances):
    if i % 1000 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        eta = (len(utterances) - i) / rate / 60 if rate > 0 else 0
        print(f'  Progress: {i}/{len(utterances)} ({i/len(utterances)*100:.1f}%) - ETA: {eta:.1f} min')
    
    vid = u['video_id']
    start = u['start']
    end = u['end']
    label = u['label']
    
    uid = f'{vid}_{start:.2f}'
    
    waveform, sr = load_audio_segment(vid, start, end, audio_files)
    
    if waveform is not None:
        emb = extract_wavlm(waveform, model, device)
        pros = extract_prosody(audio_files[vid], start, end, sr)
        
        all_embeddings.append(emb)
        all_prosody.append(pros)
        all_labels.append(label)
        all_uids.append(uid)
    else:
        errors += 1

elapsed = time.time() - start_time
print(f'\nExtraction complete in {elapsed/60:.1f} min')
print(f'Success: {len(all_uids)} utterances')
print(f'Errors: {errors}')

## Step 8: Convert to Arrays and Save

In [ ]:
embeddings = np.array(all_embeddings, dtype=np.float32)
prosody = np.array(all_prosody, dtype=np.float32)
labels = np.array(all_labels, dtype=np.int64)
uids = np.array(all_uids)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Prosody shape: {prosody.shape}')
print(f'Labels shape: {labels.shape}')
print(f'Positive rate: {np.sum(labels==1)/len(labels)*100:.2f}%')

# Save to Google Drive
output_dir = Path('/content/gdrive/MyDrive/wavlm_prosody_555_complete')
output_dir.mkdir(exist_ok=True, parents=True)

output_file = output_dir / 'wavlm_prosody_complete.npz'
np.savez_compressed(output_file, 
                    embeddings=embeddings, 
                    prosody=prosody, 
                    labels=labels, 
                    uids=uids)

print(f'\nSaved to: {output_file}')

## Step 9: Train Model (Sanity Check)

In [ ]:
print('Running quick training sanity check...')

# Normalize prosody
prosody_scaler = StandardScaler()
prosody_scaled = prosody_scaler.fit_transform(prosody)

# Split
indices = np.arange(len(utterances))
train_idx, test_idx = train_test_split(indices, test_size=0.15, stratify=labels, random_state=42)

X_train_emb = embeddings[train_idx]
X_train_pros = prosody_scaled[train_idx]
y_train = labels[train_idx]

X_test_emb = embeddings[test_idx]
X_test_pros = prosody_scaled[test_idx]
y_test = labels[test_idx]

print(f'Train: {len(y_train)} (pos: {np.sum(y_train==1)}, {np.sum(y_train==1)/len(y_train)*100:.1f}%)')
print(f'Test: {len(y_test)} (pos: {np.sum(y_test==1)}, {np.sum(y_test==1)/len(y_test)*100:.1f}%)')

In [ ]:
# Model
class WavLMProsofyClassifier(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21, hidden_dim=128):
        super().__init__()
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_out = self.prosody_proj(prosody)
        combined = torch.cat([wavlm_emb, prosody_out], dim=1)
        return self.classifier(combined)

model = WavLMProsofyClassifier().to(device)

# Training setup
pos_weight = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

In [ ]:
# Quick training (5 epochs)
X_train_emb_t = torch.tensor(X_train_emb, dtype=torch.float32).to(device)
X_train_pros_t = torch.tensor(X_train_pros, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)

X_test_emb_t = torch.tensor(X_test_emb, dtype=torch.float32).to(device)
X_test_pros_t = torch.tensor(X_test_pros, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

print('Training for 5 epochs...')
for epoch in range(5):
    model.train()
    
    perm = torch.randperm(len(y_train_t))
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i+BATCH]
        
        optimizer.zero_grad()
        out = model(X_train_emb_t[idx], X_train_pros_t[idx])
        loss = criterion(out, y_train_t[idx])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    
    scheduler.step()
    
    model.eval()
    with torch.no_grad():
        preds = model(X_test_emb_t, X_test_pros_t).argmax(dim=1)
        f1 = f1_score(y_test_t.cpu().numpy(), preds.cpu().numpy())
    
    print(f'Epoch {epoch+1}: Test F1 = {f1:.4f}')

print('\n✅ Extraction and training complete!')